In [1]:
%matplotlib inline
# Cell 1 — parameters
SPREAD_THRESHOLD        = 20    # percentile-point spread below which a variable is 'concentrated'
P_LOW                   = 10    # lower percentile for spread calculation
P_HIGH                  = 90    # upper percentile for spread calculation
ZERO_FRACTION_THRESHOLD = 0.20  # must match step 2; from work order 2026-06-14
ZERO_COVERAGE_THRESHOLD = 0.90  # buffer weight-at-zero fraction → 'outside_active_domain'

In [2]:
# Cell 2 — imports and load Step 2 outputs
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '../../../..')
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'areas'

raw_df    = pd.read_csv(OUT / 'step2_raw.tsv',    sep='\t', index_col='hybas_id')
matrix_df = pd.read_csv(OUT / 'step2_matrix.tsv', sep='\t', index_col='hybas_id')
meta_df   = pd.read_csv(OUT / 'step2_meta.tsv',   sep='\t', index_col='api_key')

print(f'raw_df    : {raw_df.shape}')
print(f'matrix_df : {matrix_df.shape}')
print(f'meta_df   : {meta_df.shape}')

raw_df    : (9, 55)
matrix_df : (9, 54)
meta_df   : (54, 8)


In [3]:
# Cell 3 — Step 3.1: join weights onto the matrix
#
# Both files use hybas_id as index; coerce to int before joining so float64
# representations (1060041510.0 vs 1060041510) don't cause mismatches.
# Inner join: any basin missing from either side is a mismatch — fail loudly.

weights = raw_df[['weight']].copy()
weights.index = weights.index.astype(int)

matrix = matrix_df.copy()
matrix.index = matrix.index.astype(int)

joined = weights.join(matrix, how='inner')

n_expected = len(weights)
n_joined   = len(joined)
weight_sum = joined['weight'].sum()

print(f'Basins in weights : {n_expected}')
print(f'Basins after join : {n_joined}  ({"OK" if n_joined == n_expected else "MISMATCH"})')
print(f'Weight sum        : {weight_sum:.6f}  ({"OK" if abs(weight_sum - 1.0) < 0.001 else "CHECK"})')

if n_joined != n_expected:
    missing = set(weights.index) - set(matrix.index)
    print(f'Missing hybas_ids: {missing}')

Basins in weights : 9
Basins after join : 9  (OK)
Weight sum        : 1.000000  (OK)


In [4]:
# Cell 4 — Step 3.2: select continental-gradient continuous variables
#
# typology_cluster dispatch — block 1 handles this family only.
# Other clusters (scale-dependent, network-topology, local-anomaly) and
# categoricals/flags are later blocks.

block1_vars = meta_df[
    (meta_df['kind'] == 'continuous') &
    (meta_df['typology_cluster'] == 'continental-gradient')
].index.tolist()

# Keep only those present in the joined matrix
block1_vars = [v for v in block1_vars if v in joined.columns]

print(f'Block 1 variables ({len(block1_vars)}):');
for v in block1_vars:
    row = meta_df.loc[v]
    print(f'  {v:35s}  band={row["band"]}  method={row["position_method"]}')

Block 1 variables (19):
  elev_min                             band=A  method=percentile
  permafrost_extent                    band=C  method=percentile
  runoff                               band=B  method=percentile
  pct_clay                             band=B  method=percentile
  pct_clay_upstream                    band=B  method=percentile
  pct_silt                             band=B  method=percentile
  pct_silt_upstream                    band=B  method=percentile
  pct_sand                             band=B  method=percentile
  pct_sand_upstream                    band=B  method=percentile
  temp_yr                              band=C  method=percentile
  temp_yr_upstream                     band=C  method=percentile
  precip_yr                            band=C  method=percentile
  precip_yr_upstream                   band=C  method=percentile
  aridity                              band=C  method=log_percentile
  aridity_upstream                     band=C  method=log_perc

In [5]:
# Cell 5 — Step 3.3: weighted score distributions
#
# For each block-1 variable:
#   1. Pair scores with weights; drop basins where score is null.
#   2. Renormalize surviving weights to sum to 1.
#      (Null = data absence for that basin, not geographic absence;
#       renormalize rather than report a shortfall.)
#   3. Record coverage: how many basins and how much original weight contributed.
#   4. Record weight_at_zero: fraction of buffer weight with score exactly 0.0
#      (used by degenerate-at-floor guard in Cell 7).

distributions = {}   # api_key → {scores, weights, n, coverage_weight, weight_at_zero}

for var in block1_vars:
    col = joined[var].apply(pd.to_numeric, errors='coerce')
    w   = joined['weight']

    mask   = col.notna()
    scores = col[mask].values.astype(float)
    wts    = w[mask].values.astype(float)

    coverage_weight     = wts.sum()
    wts_norm            = wts / coverage_weight
    weight_at_zero_frac = float(wts[scores == 0.0].sum()) if scores.size > 0 else 0.0

    distributions[var] = {
        'scores':          scores,
        'weights':         wts_norm,
        'n':               int(mask.sum()),
        'coverage_weight': round(float(coverage_weight), 4),
        'weight_at_zero':  round(weight_at_zero_frac, 4),
    }

print(f'Distributions assembled for {len(distributions)} variables')
dropped = [(v, d) for v, d in distributions.items() if d['n'] < len(joined)]
if dropped:
    print('Variables with null-dropped basins:')
    for v, d in dropped:
        print(f'  {v}: {d["n"]}/{len(joined)} basins, coverage_weight={d["coverage_weight"]}')
else:
    print('No null-dropped basins in block-1 variables')

waz_hits = [(v, d['weight_at_zero']) for v, d in distributions.items() if d['weight_at_zero'] > 0]
if waz_hits:
    print(f'\nVariables with buffer weight at score=0:')
    for v, waz in sorted(waz_hits, key=lambda x: -x[1]):
        print(f'  {v:35s}  weight_at_zero={waz:.3f}')

Distributions assembled for 19 variables
Variables with null-dropped basins:
  pct_clay: 8/9 basins, coverage_weight=0.8372
  pct_clay_upstream: 8/9 basins, coverage_weight=0.8372
  pct_silt: 8/9 basins, coverage_weight=0.8372
  pct_silt_upstream: 8/9 basins, coverage_weight=0.8372
  pct_sand: 8/9 basins, coverage_weight=0.8372
  pct_sand_upstream: 8/9 basins, coverage_weight=0.8372

Variables with buffer weight at score=0:
  permafrost_extent                    weight_at_zero=1.000
  dist_sink                            weight_at_zero=0.465
  pasture_extent                       weight_at_zero=0.440
  pasture_extent_upstream              weight_at_zero=0.440
  pct_clay_upstream                    weight_at_zero=0.277
  pct_silt_upstream                    weight_at_zero=0.277


In [6]:
# Cell 6 — Step 3.4: coherence statistics
#
# For each variable:
#   weighted_mean = sum(score_i * weight_i)
#   weighted p10, p90 via sorted cumulative weights + linear interpolation
#   spread = p90 - p10  (in percentile points)

def weighted_quantile(scores, weights, q):
    """Weighted quantile via sorted cumulative weights, linear interpolation."""
    sort_idx = np.argsort(scores)
    s = scores[sort_idx]
    w = weights[sort_idx]
    cumw = np.cumsum(w)
    cumw /= cumw[-1]   # ensure sums to 1 after float rounding
    return float(np.interp(q, cumw, s))

stats = {}
for var, d in distributions.items():
    s, w = d['scores'], d['weights']
    wmean = float(np.dot(s, w))
    p10   = weighted_quantile(s, w, P_LOW  / 100)
    p90   = weighted_quantile(s, w, P_HIGH / 100)
    spread = p90 - p10
    stats[var] = {
        'weighted_mean':    round(wmean,  2),
        'p10':              round(p10,    2),
        'p90':              round(p90,    2),
        'spread':           round(spread, 2),
        'n':                d['n'],
        'coverage_weight':  d['coverage_weight'],
    }

stats_df = pd.DataFrame(stats).T.sort_values('spread')
print(f'Coherence statistics — block 1 ({len(stats_df)} variables)')
print(f'Spread threshold T = {SPREAD_THRESHOLD} percentile points')
print()
display(stats_df)

,weighted_mean,p10,p90,spread,n,coverage_weight
permafrost_extent,0.00,0.00,0.00,0.00,9.0,1.0000
temp_yr,97.85,96.29,98.32,2.03,9.0,1.0000
elev_min,63.20,62.02,64.16,2.13,9.0,1.0000
aridity,10.19,6.26,13.40,7.13,9.0,1.0000
temp_yr_upstream,96.45,91.08,99.11,8.03,9.0,1.0000
precip_yr,16.11,9.04,22.86,13.83,9.0,1.0000
pct_sand,80.50,69.32,89.62,20.30,8.0,0.8372
runoff,20.49,10.50,30.99,20.49,9.0,1.0000
pct_clay,26.97,7.48,35.37,27.89,8.0,0.8372
pct_silt,23.72,4.54,34.13,29.59,8.0,0.8372


In [7]:
# Cell 7 — Step 3.5: classify and emit block-1 results
#
# For each variable:
#   1. Degenerate-at-floor guard (zero-inflated variables only):
#      if zero_fraction >= ZERO_FRACTION_THRESHOLD AND weight_at_zero >= ZERO_COVERAGE_THRESHOLD
#      → verdict = 'outside_active_domain' (suppress mean; variable does not apply here)
#   2. spread < T  → 'concentrated': report weighted_mean
#   3. spread >= T → 'spread':       no mean; distribution output placeholder (later block)

results = []
for var, s in stats.items():
    # Degenerate-at-floor guard
    zf = None
    if var in meta_df.index and 'zero_fraction' in meta_df.columns:
        try:
            zf = float(meta_df.loc[var, 'zero_fraction'])
            if np.isnan(zf):
                zf = None
        except (TypeError, ValueError):
            zf = None

    waz = distributions[var].get('weight_at_zero', 0.0)

    if zf is not None and zf >= ZERO_FRACTION_THRESHOLD and waz >= ZERO_COVERAGE_THRESHOLD:
        verdict = 'outside_active_domain'
        wmean   = None
    elif s['spread'] < SPREAD_THRESHOLD:
        verdict = 'concentrated'
        wmean   = s['weighted_mean']
    else:
        verdict = 'spread'
        wmean   = None

    results.append({
        'variable':        var,
        'verdict':         verdict,
        'weighted_mean':   wmean,
        'spread':          s['spread'],
        'p10':             s['p10'],
        'p90':             s['p90'],
        'n_basins':        s['n'],
        'coverage_weight': s['coverage_weight'],
        'weight_at_zero':  waz,
    })

results_df = pd.DataFrame(results).set_index('variable').sort_values('spread')

outside      = results_df[results_df['verdict'] == 'outside_active_domain']
concentrated = results_df[results_df['verdict'] == 'concentrated']
spread_vars  = results_df[results_df['verdict'] == 'spread']

print(f'Block 1 results — T = {SPREAD_THRESHOLD}')
print(f'  outside_active_domain : {len(outside)}')
print(f'  concentrated          : {len(concentrated)}')
print(f'  spread                : {len(spread_vars)}')
print()
if len(outside):
    print('=== OUTSIDE ACTIVE DOMAIN (zero-inflated; buffer at floor) ===')
    display(outside[['spread','p10','p90','weight_at_zero','n_basins']])
    print()
print('=== CONCENTRATED (mean reported) ===')
display(concentrated[['weighted_mean','spread','p10','p90','n_basins','coverage_weight']])
print()
print('=== SPREAD (no mean — distribution output placeholder) ===')
display(spread_vars[['spread','p10','p90','n_basins','coverage_weight','weight_at_zero']])

results_df.to_csv(OUT / 'step3_block1_results.tsv', sep='\t', float_format='%.2f')
print(f'\nSaved step3_block1_results.tsv')


Saved step3_block1_results.tsv
